#  QLoRA Fine-Tuning on MedQuAD
## Fine-Tuned LLM vs. RAG — Portfolio Project

**What this notebook does:**
1. Installs Unsloth (faster LoRA on free Colab T4)
2. Downloads the MedQuAD medical Q&A dataset
3. Fine-tunes Llama-3.2-3B-Instruct with QLoRA
4. Saves the adapter to Google Drive
5. Runs a quick inference test

**Runtime:** ~60–120 min on a free Colab T4 GPU

**Before running:** Make sure GPU is enabled:
> Runtime → Change runtime type → T4 GPU

In [ ]:
# Verify GPU is available
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('No GPU detected. Enable GPU: Runtime → Change runtime type → T4 GPU')
else:
    print(' GPU detected!')

## Step 1: Install Unsloth + Dependencies

Unsloth is ~2× faster than vanilla PEFT on the same GPU — critical for finishing
fine-tuning within a Colab session.

In [ ]:
# Unsloth Colab installation — must be run before any other imports
import subprocess, sys

print('Installing Unsloth (this takes 2-3 minutes)...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    '"unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"',
    '-q'
], shell=False)

# Other dependencies
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'datasets', 'trl', 'peft', 'accelerate', 'bitsandbytes',
    'wandb', 'jsonlines', 'rich', '-q'
], shell=False)

print(' Installation complete')

## Step 2: Configuration

Fill in your API keys below. These are stored only in this session — not committed to git.

In [ ]:
import os

# ─── Fill in your credentials ──────────────────────────────────────────
HF_TOKEN    = ''   # https://huggingface.co/settings/tokens (needs Llama access)
WANDB_KEY   = ''   # https://wandb.ai/settings (optional but recommended)
# ───────────────────────────────────────────────────────────────────────

os.environ['HF_TOKEN']         = HF_TOKEN
os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN

if WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)
    print('✅ W&B logged in')
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print('ℹ️  W&B disabled (no key provided)')

# ── Hyperparameters ────────────────────────────────────────────────────
CONFIG = {
    'model_name':          'unsloth/Llama-3.2-3B-Instruct',
    'max_seq_length':      2048,
    'lora_r':              16,
    'lora_alpha':          32,
    'lora_dropout':        0.05,
    'epochs':              3,
    'batch_size':          2,
    'grad_accumulation':   4,    # effective batch = 8
    'learning_rate':       2e-4,
    'max_samples':         10000,  # reduce to 2000 for a quick test run
    'output_dir':          '/content/drive/MyDrive/medquad-lora',
}
print('Configuration set ✅')
print(CONFIG)

## Step 3: Mount Google Drive

Colab sessions disconnect after ~12 hours. Saving the checkpoint to Drive
means you don't lose the trained adapter if the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print(f"✅ Drive mounted. Checkpoints will save to: {CONFIG['output_dir']}")

## Step 4: Load & Prepare Dataset

We use the medical Q&A dataset from HuggingFace. Each Q&A pair is formatted
into the Llama-3 chat template that the base model already understands.

In [ ]:
from datasets import load_dataset
import random

print('Loading dataset...')
ds = load_dataset('lavita/ChatDoctor-HealthCareMagic-100k', split='train')
print(f'Raw dataset: {len(ds):,} samples')

# ── Format as Llama-3 chat ─────────────────────────────────────────────
SYSTEM_PROMPT = (
    'You are a knowledgeable and careful medical assistant. '
    'Answer the medical question accurately and concisely, '
    'based only on established medical knowledge. '
    'If you are uncertain, say so clearly rather than guessing.'
)

def format_sample(row):
    q = (row.get('input') or row.get('question') or '').strip()
    a = (row.get('output') or row.get('answer') or '').strip()
    if not q or not a or len(a) < 30:
        return None
    return {
        'text': (
            '<|begin_of_text|>'
            '<|start_header_id|>system<|end_header_id|>\n\n'
            f'{SYSTEM_PROMPT}<|eot_id|>'
            '<|start_header_id|>user<|end_header_id|>\n\n'
            f'{q}<|eot_id|>'
            '<|start_header_id|>assistant<|end_header_id|>\n\n'
            f'{a}<|eot_id|>'
        )
    }

# Apply formatting and filter nulls
formatted = []
for row in ds:
    result = format_sample(row)
    if result:
        formatted.append(result)

# Limit and shuffle
random.seed(42)
random.shuffle(formatted)
formatted = formatted[:CONFIG['max_samples']]

# 80/10/10 split
n = len(formatted)
n_train = int(n * 0.8)
n_val   = int(n * 0.1)
train_data = formatted[:n_train]
val_data   = formatted[n_train:n_train + n_val]
test_data  = formatted[n_train + n_val:]

from datasets import Dataset
train_ds = Dataset.from_list(train_data)
val_ds   = Dataset.from_list(val_data)

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_data):,}')
print('\nSample:')
print(train_ds[0]['text'][:400])

## Step 5: Load Model + Apply QLoRA

Unsloth loads Llama-3.2-3B in 4-bit (QLoRA) and patches it for faster training.
Only the LoRA adapter weights (~0.5% of total params) are trainable.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['model_name'],
    max_seq_length=CONFIG['max_seq_length'],
    dtype=None,           # auto-detect (bf16 on T4/A100)
    load_in_4bit=True,    # QLoRA quantisation
    token=os.environ.get('HF_TOKEN'),
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG['lora_r'],
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} = {100*trainable/total:.2f}%')

## Step 6: Train

~60–90 min on Colab T4. Training loss should drop from ~2.0 to ~0.8.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import time

training_args = TrainingArguments(
    output_dir=CONFIG['output_dir'],
    per_device_train_batch_size=CONFIG['batch_size'],
    per_device_eval_batch_size=CONFIG['batch_size'],
    gradient_accumulation_steps=CONFIG['grad_accumulation'],
    num_train_epochs=CONFIG['epochs'],
    learning_rate=CONFIG['learning_rate'],
    warmup_steps=50,
    lr_scheduler_type='cosine',
    bf16=True,
    logging_steps=10,
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    evaluation_strategy='steps',
    report_to='wandb' if os.environ.get('WANDB_DISABLED') != 'true' else 'none',
    run_name='medquad-qlora-colab',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=CONFIG['max_seq_length'],
    packing=True,
    args=training_args,
)

print('Starting training...')
start = time.time()
stats = trainer.train()
elapsed = time.time() - start

print(f'\n✅ Training complete in {elapsed/60:.1f} min')
print(f'Final training loss: {stats.training_loss:.4f}')

## Step 7: Save Adapter to Drive

In [ ]:
model.save_pretrained(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])

import os
files = os.listdir(CONFIG['output_dir'])
print(f"✅ Saved to {CONFIG['output_dir']}:")
for f in files:
    size = os.path.getsize(os.path.join(CONFIG['output_dir'], f)) / 1024**2
    print(f"  {f:40s} {size:.1f} MB")

## Step 8: Quick Inference Test

Verify the fine-tuned model produces reasonable medical answers.

In [ ]:
FastLanguageModel.for_inference(model)

test_questions = [
    'What are the main symptoms of Type 2 diabetes?',
    'How is hypertension diagnosed?',
    'What is the first-line treatment for mild depression?',
]

SYSTEM_PROMPT = (
    'You are a knowledgeable and careful medical assistant. '
    'Answer the medical question accurately and concisely.'
)

for question in test_questions:
    prompt = (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n\n'
        f'{SYSTEM_PROMPT}<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n\n'
        f'{question}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )

    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=True,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id,
    )
    n_input = inputs['input_ids'].shape[1]
    answer = tokenizer.decode(outputs[0][n_input:], skip_special_tokens=True)

    print(f'\n{"="*60}')
    print(f'Q: {question}')
    print(f'A: {answer.strip()}')



**What you've done:**
- Fine-tuned Llama-3.2-3B on medical Q&A with QLoRA
- Adapter saved to Google Drive
- Training tracked in W&B

**Next — Day 2:**
1. Run `python rag/build_index.py` to build the FAISS index
2. Run `python evaluation/run_ragas.py` to compare both pipelines

**Download your adapter:**
The adapter files in Google Drive are all you need:
- `adapter_config.json`
- `adapter_model.safetensors`
- `tokenizer.json` (and related files)

Download and put them in `checkpoints/medquad-lora/` in your repo.